In [1]:
!pip install lightgbm catboost scikit-learn pandas numpy scipy -q



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import find_peaks
from scipy.fft import rfft, rfftfreq
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.ensemble import ExtraTreesClassifier
import lightgbm as lgb
import catboost as cb
import os
from collections import Counter

# ── Configuration ──────────────────────────────────────────────────────
DATA_DIR = '/Users/dayana/git repo/machine-learning-class/lab6/knu-2026-machine-learning-final-assignment'
SEED = 42
N_FOLDS = 5

# p6: LGBM num_leaves increased 127 → 255 (more expressive, matches dataset size)
LGBM_PARAMS = {
    'objective': 'multiclass',
    'num_class': 9,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 255,           # p6: was 127
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'verbose': -1,
    'n_jobs': -1,
    'seed': SEED,
}

# p6: CatBoost — same as p5 but direction passed as cat_feature (see training loop)
CAT_PARAMS = {
    'iterations': 4000,
    'learning_rate': 0.05,
    'depth': 10,
    'l2_leaf_reg': 3,
    'loss_function': 'MultiClass',
    'eval_metric': 'TotalF1:average=Macro',
    'early_stopping_rounds': 200,
    'random_seed': SEED,
    'verbose': 0,
    'allow_writing_files': False,
}

# p6: Binary Stage 1 params — same as p5
CAT_BINARY_PARAMS = {
    'iterations': 2000,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 3,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'early_stopping_rounds': 150,
    'random_seed': SEED,
    'verbose': 0,
    'allow_writing_files': False,
}

LGBM_BINARY_PARAMS = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'verbose': -1,
    'n_jobs': -1,
    'seed': SEED,
}

# p6 NEW: ExtraTrees params — class_weight='balanced' for imbalance
ET_PARAMS = {
    'n_estimators': 500,
    'max_features': 'sqrt',
    'min_samples_leaf': 5,
    'n_jobs': -1,
    'random_state': SEED,
    'class_weight': 'balanced',
}

print("Config loaded.")


Config loaded.


In [3]:
# ── Load Raw Data ──────────────────────────────────────────────────────
# Identical to p5
print("Loading sensor files (this may take ~1-2 minutes)...")
train_accel = pd.read_csv(os.path.join(DATA_DIR, 'train-accel.csv'))
train_gyro  = pd.read_csv(os.path.join(DATA_DIR, 'train-gyro.csv'))
train_label = pd.read_csv(os.path.join(DATA_DIR, 'train-label.csv'))
test_accel  = pd.read_csv(os.path.join(DATA_DIR, 'test-accel.csv'))
test_gyro   = pd.read_csv(os.path.join(DATA_DIR, 'test-gyro.csv'))
test_label  = pd.read_csv(os.path.join(DATA_DIR, 'test-label.csv'))

print(f"train_accel : {train_accel.shape}")
print(f"train_gyro  : {train_gyro.shape}")
print(f"train_label : {train_label.shape}")
print(f"test_accel  : {test_accel.shape}")
print(f"test_gyro   : {test_gyro.shape}")
print(f"test_label  : {test_label.shape}")

# ── Device Normalization + Unknown Device Fallback — identical to p5 ───
def normalize_by_device(df_train, df_test, axes=['x', 'y', 'z']):
    """
    Z-score normalize sensor axes per device using train statistics.
    Unknown/unseen test devices fall back to global train mean/std.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()
    global_stats = {}
    for ax in axes:
        global_stats[ax] = (df_train[ax].mean(), df_train[ax].std() + 1e-8)
    known_devices = set(df_train['device'].unique())
    for device in known_devices:
        tr_mask = df_train['device'] == device
        te_mask = df_test['device'] == device
        for ax in axes:
            mu  = df_train.loc[tr_mask, ax].mean()
            sig = df_train.loc[tr_mask, ax].std() + 1e-8
            df_train.loc[tr_mask, ax] = (df_train.loc[tr_mask, ax] - mu) / sig
            df_test.loc[te_mask, ax]  = (df_test.loc[te_mask, ax]  - mu) / sig
    unknown_mask = ~df_test['device'].isin(known_devices)
    if unknown_mask.sum() > 0:
        print(f"  Applying global fallback normalization to {unknown_mask.sum()} rows "
              f"with unseen device(s): {df_test.loc[unknown_mask, 'device'].unique()}")
        for ax in axes:
            mu, sig = global_stats[ax]
            df_test.loc[unknown_mask, ax] = (df_test.loc[unknown_mask, ax] - mu) / sig
    return df_train, df_test

print("Normalizing accel by device...")
train_accel, test_accel = normalize_by_device(train_accel, test_accel)
print("Normalizing gyro by device...")
train_gyro,  test_gyro  = normalize_by_device(train_gyro, test_gyro)
print("Device normalization done.")
print(f"  Devices in train_accel: {train_accel['device'].unique()}")
print(f"  Devices in test_accel : {test_accel['device'].unique()}")


Loading sensor files (this may take ~1-2 minutes)...
train_accel : (2428374, 7)
train_gyro  : (2433673, 7)
train_label : (38015, 4)
test_accel  : (2528310, 7)
test_gyro   : (2541831, 7)
test_label  : (39473, 4)
Normalizing accel by device...
  Applying global fallback normalization to 92768 rows with unseen device(s): <StringArray>
['unknown']
Length: 1, dtype: str
Normalizing gyro by device...
  Applying global fallback normalization to 92737 rows with unseen device(s): <StringArray>
['unknown']
Length: 1, dtype: str
Device normalization done.
  Devices in train_accel: <StringArray>
['samsung', 'Apple']
Length: 2, dtype: str
  Devices in test_accel : <StringArray>
['unknown', 'samsung', 'Apple']
Length: 3, dtype: str


In [4]:
# ── Direction-Corrected Axes — identical to p5 ─────────────────────────
def apply_direction_correction(df):
    """
    Rotate x, y, z into canonical body frame based on phone direction.
    Adds columns: xb (lateral), yb (forward/back), zb (vertical), magb.
    """
    df = df.copy()
    df['xb'] = df['x'].copy()
    df['yb'] = df['y'].copy()
    df['zb'] = df['z'].copy()
    for d, (sx, sy, sz) in {
        1: ( 'y', '-x',  'z'),
        2: ('-y',  'x',  'z'),
        3: ( 'y', '-x', '-z'),
        4: ('-y',  'x', '-z'),
    }.items():
        mask = df['direction'] == d
        df.loc[mask, 'xb'] = (df.loc[mask, 'x'] if sx == 'x'
                               else -df.loc[mask, 'x'] if sx == '-x'
                               else  df.loc[mask, 'y'] if sx == 'y'
                               else -df.loc[mask, 'y'])
        df.loc[mask, 'yb'] = (df.loc[mask, 'x'] if sy == 'x'
                               else -df.loc[mask, 'x'] if sy == '-x'
                               else  df.loc[mask, 'y'] if sy == 'y'
                               else -df.loc[mask, 'y'])
        df.loc[mask, 'zb'] = (df.loc[mask, 'z'] if sz == 'z'
                               else -df.loc[mask, 'z'])
    df['magb'] = np.sqrt(df['xb']**2 + df['yb']**2 + df['zb']**2)
    return df

print("Applying direction correction to accel...")
train_accel = apply_direction_correction(train_accel)
test_accel  = apply_direction_correction(test_accel)
print("Applying direction correction to gyro...")
train_gyro  = apply_direction_correction(train_gyro)
test_gyro   = apply_direction_correction(test_gyro)
print("Direction correction done.")

Applying direction correction to accel...
Applying direction correction to gyro...
Direction correction done.


In [5]:
# ── Feature Extraction Helpers — identical to p5 ───────────────────────
FS = 50.0  # nominal sampling rate in Hz

def stat_features(arr, prefix):
    feats = {}
    n = len(arr)
    if n == 0:
        for k in ['mean','std','var','min','max','range','median',
                  'p25','p75','iqr','skew','kurt','rms','energy',
                  'zero_cross','mean_abs_diff','max_abs']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    feats[f'{prefix}_mean']          = np.mean(arr)
    feats[f'{prefix}_std']           = np.std(arr)
    feats[f'{prefix}_var']           = np.var(arr)
    feats[f'{prefix}_min']           = np.min(arr)
    feats[f'{prefix}_max']           = np.max(arr)
    feats[f'{prefix}_range']         = np.ptp(arr)
    feats[f'{prefix}_median']        = np.median(arr)
    feats[f'{prefix}_p25']           = np.percentile(arr, 25)
    feats[f'{prefix}_p75']           = np.percentile(arr, 75)
    feats[f'{prefix}_iqr']           = np.percentile(arr, 75) - np.percentile(arr, 25)
    feats[f'{prefix}_skew']          = float(stats.skew(arr)) if n > 2 else 0.0
    feats[f'{prefix}_kurt']          = float(stats.kurtosis(arr)) if n > 2 else 0.0
    feats[f'{prefix}_rms']           = np.sqrt(np.mean(arr**2))
    feats[f'{prefix}_energy']        = np.sum(arr**2) / n
    feats[f'{prefix}_zero_cross']    = np.sum(np.diff(np.sign(arr - np.mean(arr))) != 0)
    feats[f'{prefix}_mean_abs_diff'] = np.mean(np.abs(np.diff(arr))) if n > 1 else 0.0
    feats[f'{prefix}_max_abs']       = np.max(np.abs(arr))
    return feats

def fft_features(arr, prefix, fs=FS):
    feats = {}
    n = len(arr)
    if n < 8:
        for k in ['dom_freq','dom_amp','spectral_entropy',
                  'band_low','band_mid','band_high','band_ratio_mid_low']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    arr_centered  = arr - np.mean(arr)
    fft_vals      = np.abs(rfft(arr_centered))
    freqs         = rfftfreq(n, d=1.0/fs)
    total_power   = np.sum(fft_vals**2) + 1e-10
    dom_idx       = np.argmax(fft_vals)
    feats[f'{prefix}_dom_freq'] = freqs[dom_idx]
    feats[f'{prefix}_dom_amp']  = fft_vals[dom_idx]
    psd_norm = fft_vals**2 / total_power
    psd_norm = psd_norm[psd_norm > 0]
    feats[f'{prefix}_spectral_entropy'] = -np.sum(psd_norm * np.log(psd_norm))
    low_mask  = (freqs >= 0.0) & (freqs < 1.0)
    mid_mask  = (freqs >= 1.0) & (freqs < 3.0)
    high_mask = (freqs >= 3.0) & (freqs < 8.0)
    feats[f'{prefix}_band_low']          = np.sum(fft_vals[low_mask]**2)  / total_power
    feats[f'{prefix}_band_mid']          = np.sum(fft_vals[mid_mask]**2)  / total_power
    feats[f'{prefix}_band_high']         = np.sum(fft_vals[high_mask]**2) / total_power
    feats[f'{prefix}_band_ratio_mid_low'] = (
        feats[f'{prefix}_band_mid'] / (feats[f'{prefix}_band_low'] + 1e-10)
    )
    return feats

def peak_features(arr, prefix, fs=FS):
    feats = {}
    n = len(arr)
    if n < 4:
        for k in ['n_peaks','peak_rate','mean_peak_height',
                  'std_peak_height','mean_peak_interval','peak_regularity']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    arr_centered   = arr - np.mean(arr)
    height_thresh  = 0.3 * np.std(arr_centered)
    peaks, props   = find_peaks(arr_centered, height=height_thresh,
                                distance=int(fs * 0.15))
    n_peaks        = len(peaks)
    duration_s     = n / fs
    feats[f'{prefix}_n_peaks']    = n_peaks
    feats[f'{prefix}_peak_rate']  = n_peaks / duration_s
    if n_peaks > 0:
        ph = props['peak_heights']
        feats[f'{prefix}_mean_peak_height'] = np.mean(ph)
        feats[f'{prefix}_std_peak_height']  = np.std(ph)
    else:
        feats[f'{prefix}_mean_peak_height'] = 0.0
        feats[f'{prefix}_std_peak_height']  = 0.0
    if n_peaks > 1:
        intervals = np.diff(peaks) / fs
        feats[f'{prefix}_mean_peak_interval'] = np.mean(intervals)
        feats[f'{prefix}_peak_regularity']    = 1.0 / (np.std(intervals) + 1e-8)
    else:
        feats[f'{prefix}_mean_peak_interval'] = 0.0
        feats[f'{prefix}_peak_regularity']    = 0.0
    return feats

def posture_features(a_xb, a_yb, a_zb):
    feats = {}
    if len(a_xb) == 0:
        feats.update({
            'posture_zb_mean': 0.0, 'posture_yb_mean': 0.0,
            'posture_tilt_angle': 0.0, 'posture_prone_score': 0.0,
            'posture_vertical_energy': 0.0,
        })
        return feats
    grav_x   = np.mean(a_xb)
    grav_y   = np.mean(a_yb)
    grav_z   = np.mean(a_zb)
    grav_mag = np.sqrt(grav_x**2 + grav_y**2 + grav_z**2) + 1e-8
    feats['posture_zb_mean']          = grav_z
    feats['posture_yb_mean']          = grav_y
    feats['posture_tilt_angle']       = np.arccos(np.clip(abs(grav_y) / grav_mag, -1, 1))
    feats['posture_prone_score']      = abs(grav_z) / grav_mag
    dynamic_zb = np.array(a_zb) - grav_z
    feats['posture_vertical_energy']  = np.mean(dynamic_zb**2)
    return feats

def jerk_features(arr, prefix, fs=FS):
    feats = {}
    keys = ['mean_abs','std','max_abs','rms','energy',
            'p90_abs','p95_abs','skew','kurt',
            'n_peaks','peak_rate','mean_peak_height','impulse_ratio']
    if len(arr) < 2:
        for k in keys: feats[f'{prefix}_{k}'] = 0.0
        return feats
    jerk = np.diff(arr.astype(np.float64)) * fs
    n    = len(jerk)
    feats[f'{prefix}_mean_abs']  = np.mean(np.abs(jerk))
    feats[f'{prefix}_std']       = np.std(jerk)
    feats[f'{prefix}_max_abs']   = np.max(np.abs(jerk))
    feats[f'{prefix}_rms']       = np.sqrt(np.mean(jerk**2))
    feats[f'{prefix}_energy']    = np.sum(jerk**2) / n
    feats[f'{prefix}_p90_abs']   = np.percentile(np.abs(jerk), 90)
    feats[f'{prefix}_p95_abs']   = np.percentile(np.abs(jerk), 95)
    feats[f'{prefix}_skew']      = float(stats.skew(jerk)) if n > 2 else 0.0
    feats[f'{prefix}_kurt']      = float(stats.kurtosis(jerk)) if n > 2 else 0.0
    abs_jerk = np.abs(jerk)
    thresh   = 0.5 * np.std(abs_jerk)
    peaks, props = find_peaks(abs_jerk, height=thresh, distance=int(fs * 0.1))
    n_peaks      = len(peaks)
    duration_s   = n / fs
    feats[f'{prefix}_n_peaks']          = n_peaks
    feats[f'{prefix}_peak_rate']        = n_peaks / duration_s
    feats[f'{prefix}_mean_peak_height'] = (
        np.mean(props['peak_heights']) if n_peaks > 0 else 0.0
    )
    mean_abs = feats[f'{prefix}_mean_abs'] + 1e-10
    feats[f'{prefix}_impulse_ratio'] = feats[f'{prefix}_max_abs'] / mean_abs
    return feats

def lateral_asymmetry_features(axb, prefix='lat'):
    feats = {}
    keys  = ['xb_skew','xb_kurt','xb_asymmetry_idx',
             'xb_pos_ratio','xb_range_ratio','xb_mean_abs']
    arr   = np.asarray(axb, dtype=np.float64)
    arr   = arr[np.isfinite(arr)]
    if len(arr) < 4:
        for k in keys: feats[f'{prefix}_{k}'] = 0.0
        return feats
    arr_c = arr - np.mean(arr)
    feats[f'{prefix}_xb_skew']         = float(stats.skew(arr_c)) if len(arr_c) > 2 else 0.0
    feats[f'{prefix}_xb_kurt']         = float(stats.kurtosis(arr_c)) if len(arr_c) > 2 else 0.0
    feats[f'{prefix}_xb_asymmetry_idx']= float(abs(np.mean(arr)) / (np.std(arr) + 1e-8))
    pos_vals = arr[arr > 0]
    feats[f'{prefix}_xb_pos_ratio']    = float(len(pos_vals) / len(arr))
    feats[f'{prefix}_xb_range_ratio']  = float(np.max(arr_c) / (abs(np.min(arr_c)) + 1e-8))
    feats[f'{prefix}_xb_mean_abs']     = float(np.mean(np.abs(arr_c)))
    return feats

def autocorr_features(arr, prefix, fs=FS, lags_s=(0.5, 1.0, 1.5, 2.0, 2.5)):
    feats    = {}
    lag_names= [str(lag).replace('.', 'p') for lag in lags_s]
    keys     = [f'lag_{name}' for name in lag_names] + ['max','mean','std','best_lag_s']
    arr      = np.asarray(arr, dtype=np.float64)
    arr      = arr[np.isfinite(arr)]
    if len(arr) < int(max(lags_s) * fs) + 2 or np.std(arr) < 1e-8:
        for k in keys: feats[f'{prefix}_{k}'] = 0.0
        return feats
    x    = arr - np.mean(arr)
    x    = x / (np.std(x) + 1e-8)
    vals = []
    for lag_s, lag_name in zip(lags_s, lag_names):
        lag = int(round(lag_s * fs))
        if lag <= 0 or len(x) <= lag:
            ac = 0.0
        else:
            ac = float(np.mean(x[:-lag] * x[lag:]))
        feats[f'{prefix}_lag_{lag_name}'] = ac
        vals.append(ac)
    vals = np.asarray(vals, dtype=np.float64)
    feats[f'{prefix}_max']         = float(np.max(vals))
    feats[f'{prefix}_mean']        = float(np.mean(vals))
    feats[f'{prefix}_std']         = float(np.std(vals))
    feats[f'{prefix}_best_lag_s']  = float(lags_s[int(np.argmax(vals))])
    return feats

print("Feature extraction helpers defined.")

Feature extraction helpers defined.


In [6]:
# ── Main Feature Extraction — p6 ───────────────────────────────────────
CONTEXT_SHORT   = 1  # ±1s for stat features
CONTEXT_LONG    = 4  # ±4s for FFT short / jerk / autocorr / peaks
CONTEXT_FFT_LONG = 6 # ±6s for extended FFT features (same as p5)

def compute_features_p6(accel_df, gyro_df, label_df):
    """
    p6 feature extraction.
    Inherits all p5 features plus:
    - Raw x, y, z stat features kept alongside body-frame (xb, yb, zb).
      This gives the model fallback signal if direction correction is
      partially wrong for some PIDs.
    All other feature groups (FFT, peaks, jerk, autocorr, interactions)
    are identical to p5.
    """
    accel_df = accel_df.copy()
    gyro_df  = gyro_df.copy()
    label_df = label_df.copy()

    accel_df['time_s'] = accel_df['time'].astype(int)
    gyro_df['time_s']  = gyro_df['time'].astype(int)
    label_df['time_s'] = label_df['time'].astype(int)

    for df in [accel_df, gyro_df]:
        df['mag']  = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

    print("  Building sensor lookup tables...")
    def build_lookup(df, cols):
        lookup = {}
        for (pid, ts), grp in df.groupby(['pid', 'time_s']):
            lookup[(pid, ts)] = grp[cols].values
        return lookup

    accel_raw_lookup  = build_lookup(accel_df, ['x','y','z','mag'])
    accel_body_lookup = build_lookup(accel_df, ['xb','yb','zb','magb'])
    gyro_raw_lookup   = build_lookup(gyro_df,  ['x','y','z','mag'])
    gyro_body_lookup  = build_lookup(gyro_df,  ['xb','yb','zb','magb'])

    meta_lookup = {}
    for (pid, ts), grp in accel_df.groupby(['pid', 'time_s']):
        meta_lookup[(pid, ts)] = {
            'direction': grp['direction'].iloc[0],
            'device':    1 if str(grp['device'].iloc[0]).lower() == 'apple' else 0,
        }

    print("  Extracting features per label row (this takes a few minutes)...")
    all_feats = []
    for idx, (_, row) in enumerate(label_df.iterrows()):
        pid = row['pid']
        ts  = int(row['time_s'])
        feat = {}

        # ── Collect SHORT window arrays (±1s) ──────────────────────────
        ax_s, ay_s, az_s, amag_s = [], [], [], []
        gx_s, gy_s, gz_s, gmag_s = [], [], [], []
        axb_s, ayb_s, azb_s      = [], [], []
        for offset in range(-CONTEXT_SHORT, CONTEXT_SHORT + 1):
            key = (pid, ts + offset)
            if key in accel_raw_lookup:
                c = accel_raw_lookup[key]
                ax_s.extend(c[:,0]); ay_s.extend(c[:,1])
                az_s.extend(c[:,2]); amag_s.extend(c[:,3])
            if key in accel_body_lookup:
                c = accel_body_lookup[key]
                axb_s.extend(c[:,0]); ayb_s.extend(c[:,1]); azb_s.extend(c[:,2])
            if key in gyro_raw_lookup:
                c = gyro_raw_lookup[key]
                gx_s.extend(c[:,0]); gy_s.extend(c[:,1])
                gz_s.extend(c[:,2]); gmag_s.extend(c[:,3])

        # ── Collect LONG window arrays (±4s) ───────────────────────────
        amag_l, axb_l, ayb_l, azb_l = [], [], [], []
        gmag_l, gxb_l, gyb_l, gzb_l = [], [], [], []
        for offset in range(-CONTEXT_LONG, CONTEXT_LONG + 1):
            key = (pid, ts + offset)
            if key in accel_raw_lookup:
                c = accel_raw_lookup[key]; amag_l.extend(c[:,3])
            if key in accel_body_lookup:
                c = accel_body_lookup[key]
                axb_l.extend(c[:,0]); ayb_l.extend(c[:,1]); azb_l.extend(c[:,2])
            if key in gyro_raw_lookup:
                c = gyro_raw_lookup[key]; gmag_l.extend(c[:,3])
            if key in gyro_body_lookup:
                c = gyro_body_lookup[key]
                gxb_l.extend(c[:,0]); gyb_l.extend(c[:,1]); gzb_l.extend(c[:,2])

        # ── Collect EXTENDED LONG window (±6s) for better FFT resolution
        amag_xl, ayb_xl, azb_xl = [], [], []
        gmag_xl, gyb_xl         = [], []
        for offset in range(-CONTEXT_FFT_LONG, CONTEXT_FFT_LONG + 1):
            key = (pid, ts + offset)
            if key in accel_raw_lookup:
                c = accel_raw_lookup[key]; amag_xl.extend(c[:,3])
            if key in accel_body_lookup:
                c = accel_body_lookup[key]
                ayb_xl.extend(c[:,1]); azb_xl.extend(c[:,2])
            if key in gyro_raw_lookup:
                c = gyro_raw_lookup[key]; gmag_xl.extend(c[:,3])
            if key in gyro_body_lookup:
                c = gyro_body_lookup[key]; gyb_xl.extend(c[:,1])

        # numpy conversions
        ax_s   = np.array(ax_s,   dtype=np.float32)
        ay_s   = np.array(ay_s,   dtype=np.float32)
        az_s   = np.array(az_s,   dtype=np.float32)
        amag_s = np.array(amag_s, dtype=np.float32)
        gx_s   = np.array(gx_s,   dtype=np.float32)
        gy_s   = np.array(gy_s,   dtype=np.float32)
        gz_s   = np.array(gz_s,   dtype=np.float32)
        gmag_s = np.array(gmag_s, dtype=np.float32)
        axb_s  = np.array(axb_s,  dtype=np.float32)
        ayb_s  = np.array(ayb_s,  dtype=np.float32)
        azb_s  = np.array(azb_s,  dtype=np.float32)

        amag_l = np.array(amag_l, dtype=np.float32)
        axb_l  = np.array(axb_l,  dtype=np.float32)
        ayb_l  = np.array(ayb_l,  dtype=np.float32)
        azb_l  = np.array(azb_l,  dtype=np.float32)
        gmag_l = np.array(gmag_l, dtype=np.float32)
        gxb_l  = np.array(gxb_l,  dtype=np.float32)
        gyb_l  = np.array(gyb_l,  dtype=np.float32)
        gzb_l  = np.array(gzb_l,  dtype=np.float32)

        amag_xl = np.array(amag_xl, dtype=np.float32)
        ayb_xl  = np.array(ayb_xl,  dtype=np.float32)
        azb_xl  = np.array(azb_xl,  dtype=np.float32)
        gmag_xl = np.array(gmag_xl, dtype=np.float32)
        gyb_xl  = np.array(gyb_xl,  dtype=np.float32)

        # ═══════════════════════════════════════════════════════════════
        # STAT FEATURES (short window ±1s)
        # p6: keeps raw x,y,z stats IN ADDITION to body-frame xb,yb,zb.
        # The raw features provide a fallback if direction correction is
        # partially wrong — the model can decide which to use.
        # ═══════════════════════════════════════════════════════════════
        feat.update(stat_features(ax_s,   'ax'))
        feat.update(stat_features(ay_s,   'ay'))
        feat.update(stat_features(az_s,   'az'))
        feat.update(stat_features(amag_s, 'amag'))
        feat.update(stat_features(axb_s,  'axb'))
        feat.update(stat_features(ayb_s,  'ayb'))
        feat.update(stat_features(azb_s,  'azb'))
        feat.update(stat_features(gx_s,   'gx'))
        feat.update(stat_features(gy_s,   'gy'))
        feat.update(stat_features(gz_s,   'gz'))
        feat.update(stat_features(gmag_s, 'gmag'))

        # Cross-sensor correlation
        if len(amag_s) > 2 and len(gmag_s) > 2:
            ml = min(len(amag_s), len(gmag_s))
            feat['accel_gyro_mag_corr'] = float(np.corrcoef(amag_s[:ml], gmag_s[:ml])[0, 1])
        else:
            feat['accel_gyro_mag_corr'] = 0.0

        # ═══════════════════════════════════════════════════════════════
        # FFT FEATURES (long window ±4s) — unchanged from p5
        # ═══════════════════════════════════════════════════════════════
        feat.update(fft_features(amag_l, 'amag_fft'))
        feat.update(fft_features(axb_l,  'axb_fft'))
        feat.update(fft_features(ayb_l,  'ayb_fft'))
        feat.update(fft_features(azb_l,  'azb_fft'))
        feat.update(fft_features(gmag_l, 'gmag_fft'))
        feat.update(fft_features(gxb_l,  'gxb_fft'))
        feat.update(fft_features(gyb_l,  'gyb_fft'))
        feat.update(fft_features(gzb_l,  'gzb_fft'))

        # ═══════════════════════════════════════════════════════════════
        # EXTENDED FFT FEATURES (±6s) — unchanged from p5
        # ═══════════════════════════════════════════════════════════════
        feat.update(fft_features(amag_xl, 'amag_xfft'))
        feat.update(fft_features(ayb_xl,  'ayb_xfft'))
        feat.update(fft_features(azb_xl,  'azb_xfft'))
        feat.update(fft_features(gmag_xl, 'gmag_xfft'))
        feat.update(fft_features(gyb_xl,  'gyb_xfft'))

        # ═══════════════════════════════════════════════════════════════
        # PEAK FEATURES (long window ±4s) — unchanged from p5
        # ═══════════════════════════════════════════════════════════════
        feat.update(peak_features(amag_l, 'amag_pk'))
        feat.update(peak_features(axb_l,  'axb_pk'))
        feat.update(peak_features(ayb_l,  'ayb_pk'))
        feat.update(peak_features(azb_l,  'azb_pk'))
        feat.update(peak_features(gmag_l, 'gmag_pk'))

        # ═══════════════════════════════════════════════════════════════
        # POSTURE FEATURES — unchanged from p5
        # ═══════════════════════════════════════════════════════════════
        feat.update(posture_features(axb_s, ayb_s, azb_s))

        # ═══════════════════════════════════════════════════════════════
        # JERK FEATURES — unchanged from p5
        # ═══════════════════════════════════════════════════════════════
        feat.update(jerk_features(amag_s, 'amag_jerk_s'))
        feat.update(jerk_features(azb_s,  'azb_jerk_s'))
        feat.update(jerk_features(gmag_s, 'gmag_jerk_s'))
        feat.update(jerk_features(amag_l, 'amag_jerk_l'))
        feat.update(jerk_features(azb_l,  'azb_jerk_l'))
        feat.update(jerk_features(gmag_l, 'gmag_jerk_l'))

        # ═══════════════════════════════════════════════════════════════
        # AUTOCORRELATION FEATURES — same 7 channels as p5
        # ═══════════════════════════════════════════════════════════════
        feat.update(autocorr_features(amag_l, 'amag_acorr_l'))
        feat.update(autocorr_features(azb_l,  'azb_acorr_l'))
        feat.update(autocorr_features(gmag_l, 'gmag_acorr_l'))
        feat.update(autocorr_features(ayb_l,  'ayb_acorr_l'))
        feat.update(autocorr_features(axb_l,  'axb_acorr_l'))
        feat.update(autocorr_features(gxb_l,  'gxb_acorr_l'))
        feat.update(autocorr_features(gyb_l,  'gyb_acorr_l'))

        # ═══════════════════════════════════════════════════════════════
        # LATERAL ASYMMETRY FEATURES — unchanged from p5
        # ═══════════════════════════════════════════════════════════════
        feat.update(lateral_asymmetry_features(axb_s, prefix='lat_s'))
        feat.update(lateral_asymmetry_features(axb_l, prefix='lat_l'))

        # ═══════════════════════════════════════════════════════════════
        # GRAVITY + TILT — unchanged from p5
        # ═══════════════════════════════════════════════════════════════
        if len(ax_s) > 0:
            feat['gravity_x']   = float(np.mean(ax_s))
            feat['gravity_y']   = float(np.mean(ay_s))
            feat['gravity_z']   = float(np.mean(az_s))
            gm = np.sqrt(feat['gravity_x']**2 + feat['gravity_y']**2 + feat['gravity_z']**2)
            feat['gravity_mag'] = gm
            feat['tilt_xz']     = float(np.arctan2(feat['gravity_x'], feat['gravity_z'] + 1e-8))
            feat['tilt_yz']     = float(np.arctan2(feat['gravity_y'], feat['gravity_z'] + 1e-8))
        else:
            feat.update({'gravity_x':0,'gravity_y':0,'gravity_z':0,
                         'gravity_mag':0,'tilt_xz':0,'tilt_yz':0})

        # Data quality
        feat['n_accel_samples'] = len(ax_s)
        feat['n_gyro_samples']  = len(gx_s)

        # Metadata
        if (pid, ts) in meta_lookup:
            feat['direction'] = meta_lookup[(pid, ts)]['direction']
            feat['device']    = meta_lookup[(pid, ts)]['device']
        else:
            feat['direction'] = -1
            feat['device']    = -1

        all_feats.append(feat)
        if (idx + 1) % 5000 == 0:
            print(f"    {idx+1}/{len(label_df)} rows done...")

    feat_df = pd.DataFrame(all_feats)
    feat_df.index = label_df.index
    return feat_df

print("p6 feature extraction function defined.")

p6 feature extraction function defined.


In [7]:
# ── Run Feature Extraction ─────────────────────────────────────────────
print("Extracting TRAIN features...")
train_feats = compute_features_p6(train_accel, train_gyro, train_label)
print(f"Train features shape: {train_feats.shape}")

print("\nExtracting TEST features...")
test_feats = compute_features_p6(test_accel, test_gyro, test_label)
print(f"Test features shape: {test_feats.shape}")

Extracting TRAIN features...
  Building sensor lookup tables...
  Extracting features per label row (this takes a few minutes)...
    5000/38015 rows done...
    10000/38015 rows done...
    15000/38015 rows done...
    20000/38015 rows done...
    25000/38015 rows done...
    30000/38015 rows done...
    35000/38015 rows done...
Train features shape: (38015, 477)

Extracting TEST features...
  Building sensor lookup tables...
  Extracting features per label row (this takes a few minutes)...
    5000/39473 rows done...
    10000/39473 rows done...
    15000/39473 rows done...
    20000/39473 rows done...
    25000/39473 rows done...
    30000/39473 rows done...
    35000/39473 rows done...
Test features shape: (39473, 477)


In [8]:
# ── Remove Per-PID Rest-Baseline Normalization — same as p5 ────────────
train_feats_clean = train_feats.copy()
test_feats_clean  = test_feats.copy()
print("Per-PID rest-baseline feature normalization: REMOVED (same as p5).")


Per-PID rest-baseline feature normalization: REMOVED (same as p5).


In [9]:
# ── p5 Ratio / Interaction Features — identical to p5 ──────────────────
def add_interaction_features(df):
    df  = df.copy()
    eps = 1e-10
    df['inter_gyb_dom_amp_over_gy_iqr'] = (
        df['gyb_fft_dom_amp'] / (df['gy_iqr'] + eps)
    )
    df['inter_ayb_band_low_over_mid'] = (
        df['ayb_fft_band_low'] / (df['ayb_fft_band_mid'] + eps)
    )
    df['inter_tilt_x_amag_std'] = (
        df['posture_tilt_angle'] * df['amag_std']
    )
    df['inter_gxb_mid_over_gmag_mid'] = (
        df['gxb_fft_band_mid'] / (df['gmag_fft_band_mid'] + eps)
    )
    df['inter_azb_acorr_x_ayb_dom_amp'] = (
        df['azb_acorr_l_best_lag_s'] * df['ayb_fft_dom_amp']
    )
    df['inter_gyb_entropy_over_amag_entropy'] = (
        df['gyb_fft_spectral_entropy'] / (df['amag_fft_spectral_entropy'] + eps)
    )
    df['inter_ayb_acorr_over_azb_acorr'] = (
        df['ayb_acorr_l_max'] / (df['azb_acorr_l_max'] + eps)
    )
    df['inter_ayb_xfft_low_over_fft_low'] = (
        df['ayb_xfft_band_low'] / (df['ayb_fft_band_low'] + eps)
    )
    return df

print("Adding interaction features to train...")
train_feats_clean = add_interaction_features(train_feats_clean)
print("Adding interaction features to test...")
test_feats_clean  = add_interaction_features(test_feats_clean)
print(f"Train features after interactions: {train_feats_clean.shape}")
print(f"Test  features after interactions: {test_feats_clean.shape}")


Adding interaction features to train...
Adding interaction features to test...
Train features after interactions: (38015, 485)
Test  features after interactions: (39473, 485)


In [10]:
train_feats_clean["device_is_unknown"] = 0

test_feats_clean["device_is_unknown"] = (
    test_feats_clean["device"]
    .astype(str)
    .str.lower()
    .eq("unknown")
    .astype(np.int8)
)

In [11]:
# ── Prepare ML Inputs ──────────────────────────────────────────────────
train_feats_model = train_feats_clean.fillna(0).replace([np.inf, -np.inf], 0)
test_feats_model  = test_feats_clean.fillna(0).replace([np.inf, -np.inf], 0)

feat_cols       = list(train_feats_model.columns)
test_feats_model = test_feats_model.reindex(columns=feat_cols, fill_value=0)


In [12]:
# p6: identify the direction column index for CatBoost cat_features
direction_col_idx = feat_cols.index('direction')
print(f"  direction column index for CatBoost: {direction_col_idx}")

X_train = train_feats_model.values.astype(np.float32)
y_train = train_label['workout'].values
groups  = train_label['pid'].values
train_times = train_label['time'].astype(int).values

X_test      = test_feats_model.values.astype(np.float32)
test_times  = test_label['time'].astype(int).values

pid71_mask      = (test_label['pid'] == '71').values
has_sensor_mask = ~pid71_mask

print(f"Feature columns : {len(feat_cols)}")
print(f"X_train shape   : {X_train.shape}")
print(f"X_test shape    : {X_test.shape}")
print(f"PID-71 rows     : {pid71_mask.sum()}")
print(f"Rows with sensor: {has_sensor_mask.sum()}")


  direction column index for CatBoost: 475
Feature columns : 486
X_train shape   : (38015, 486)
X_test shape    : (39473, 486)
PID-71 rows     : 1124
Rows with sensor: 38349


In [13]:
# ── p8: No class weights ─────────────────────────────

sample_weights_train = np.ones(
    len(y_train),
    dtype=np.float32
)

print("Using uniform sample weights")


Using uniform sample weights


In [15]:
from sklearn.model_selection import GroupKFold

N_FOLDS = 5

gkf = GroupKFold(
    n_splits=N_FOLDS
)

print("GroupKFold ready")

GroupKFold ready


In [16]:
# ── STAGE 2: 9-Class LGBM ──────────────────────────────────────────────
print("\n" + "="*60)
print("STAGE 2: 9-class LightGBM (num_leaves=255)")
print("="*60)

oof_lgbm  = np.zeros((len(X_train), 9))
pred_lgbm = np.zeros((len(X_test),  9))
lgbm_models = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    sw_tr       = sample_weights_train[tr_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr, feature_name=feat_cols)
    dval   = lgb.Dataset(X_val, label=y_val, feature_name=feat_cols, reference=dtrain)
    model  = lgb.train(
        LGBM_PARAMS, dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )
    oof_lgbm[val_idx]  = model.predict(X_val, num_iteration=model.best_iteration)
    pred_lgbm         += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS

    score      = balanced_accuracy_score(y_val, np.argmax(oof_lgbm[val_idx], axis=1))
    held_pids  = list(np.unique(groups[val_idx]))
    print(f"  Fold {fold+1} | PIDs: {held_pids} | BA: {score:.5f} | iter: {model.best_iteration}")
    lgbm_models.append(model)

lgbm_oof_ba = balanced_accuracy_score(y_train, np.argmax(oof_lgbm, axis=1))
print(f"\nLGBM OOF BA (raw, no Stage1): {lgbm_oof_ba:.5f}")



STAGE 2: 9-class LightGBM (num_leaves=255)
  Fold 1 | PIDs: ['9XOO', 'CDQ6', 'LIUY', 'MEM8', 'P3LG', 'QJ18', 'QZJ2', 'RQFN', 'TF0Y', 'UQSU', 'ZF6S'] | BA: 0.87886 | iter: 47
  Fold 2 | PIDs: ['10ZQ', '70N8', '7FRZ', '7KPX', 'CQ2G', 'DT5C', 'NQRB', 'OL6N', 'SNG7', 'TPQI', 'VBMP', 'Y21H'] | BA: 0.95018 | iter: 179
  Fold 3 | PIDs: ['2Q0J', '3C4K', '3H2A', '4UC1', 'BFIE', 'HDS9', 'IKYW', 'N6RZ', 'TGQ4', 'WZDL', 'ZL3U'] | BA: 0.87124 | iter: 58
  Fold 4 | PIDs: ['01Z2', '2XO3', '9HJO', 'D1XP', 'DU2K', 'EQZH', 'EUS2', 'F1ZM', 'IYWF', 'P4DZ', 'UWUT', 'VN8D'] | BA: 0.94670 | iter: 70
  Fold 5 | PIDs: ['13P2', '222X', '43JW', '4N6K', '7PF3', '94BI', 'C8Q6', 'J2JZ', 'MADD', 'PYXQ', 'QEYR', 'SE4Q'] | BA: 0.85683 | iter: 72

LGBM OOF BA (raw, no Stage1): 0.89982


In [17]:
# ── STAGE 2: 9-Class CatBoost (with direction as categorical) ──────────
print("\n" + "="*60)
print("STAGE 2: 9-class CatBoost (direction as categorical feature)")
print("="*60)

oof_cat  = np.zeros((len(X_train), 9))
pred_cat = np.zeros((len(X_test),  9))
cat_models = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    sw_tr       = sample_weights_train[tr_idx]

    model = cb.CatBoostClassifier(**CAT_PARAMS)
    # p6: pass direction column index as categorical to CatBoost
    model.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=(X_val, y_val), use_best_model=True, verbose=False)
    oof_cat[val_idx]  = model.predict_proba(X_val)
    pred_cat         += model.predict_proba(X_test) / N_FOLDS

    score     = balanced_accuracy_score(y_val, np.argmax(oof_cat[val_idx], axis=1))
    held_pids = list(np.unique(groups[val_idx]))
    print(f"  Fold {fold+1} | PIDs: {held_pids} | BA: {score:.5f} | iter: {model.best_iteration_}")
    cat_models.append(model)

cat_oof_ba = balanced_accuracy_score(y_train, np.argmax(oof_cat, axis=1))
print(f"\nCatBoost OOF BA (raw, no Stage1): {cat_oof_ba:.5f}")



STAGE 2: 9-class CatBoost (direction as categorical feature)
  Fold 1 | PIDs: ['9XOO', 'CDQ6', 'LIUY', 'MEM8', 'P3LG', 'QJ18', 'QZJ2', 'RQFN', 'TF0Y', 'UQSU', 'ZF6S'] | BA: 0.89289 | iter: 1362
  Fold 2 | PIDs: ['10ZQ', '70N8', '7FRZ', '7KPX', 'CQ2G', 'DT5C', 'NQRB', 'OL6N', 'SNG7', 'TPQI', 'VBMP', 'Y21H'] | BA: 0.94267 | iter: 1289
  Fold 3 | PIDs: ['2Q0J', '3C4K', '3H2A', '4UC1', 'BFIE', 'HDS9', 'IKYW', 'N6RZ', 'TGQ4', 'WZDL', 'ZL3U'] | BA: 0.88667 | iter: 1499
  Fold 4 | PIDs: ['01Z2', '2XO3', '9HJO', 'D1XP', 'DU2K', 'EQZH', 'EUS2', 'F1ZM', 'IYWF', 'P4DZ', 'UWUT', 'VN8D'] | BA: 0.95460 | iter: 1007
  Fold 5 | PIDs: ['13P2', '222X', '43JW', '4N6K', '7PF3', '94BI', 'C8Q6', 'J2JZ', 'MADD', 'PYXQ', 'QEYR', 'SE4Q'] | BA: 0.87380 | iter: 596

CatBoost OOF BA (raw, no Stage1): 0.90924


In [18]:
# ── STAGE 2: ExtraTrees — p6 NEW ───────────────────────────────────────
print("\n" + "="*60)
print("STAGE 2: ExtraTrees 9-class (p6 new model for diversity)")
print("="*60)

oof_et  = np.zeros((len(X_train), 9))
pred_et = np.zeros((len(X_test),  9))
et_models = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = ExtraTreesClassifier(**ET_PARAMS)
    model.fit(X_tr, y_tr)
    oof_et[val_idx]  = model.predict_proba(X_val)
    pred_et         += model.predict_proba(X_test) / N_FOLDS

    score     = balanced_accuracy_score(y_val, np.argmax(oof_et[val_idx], axis=1))
    held_pids = list(np.unique(groups[val_idx]))
    print(f"  Fold {fold+1} | PIDs: {held_pids} | BA: {score:.5f}")
    et_models.append(model)

et_oof_ba = balanced_accuracy_score(y_train, np.argmax(oof_et, axis=1))
print(f"\nExtraTrees OOF BA (raw, no Stage1): {et_oof_ba:.5f}")



STAGE 2: ExtraTrees 9-class (p6 new model for diversity)
  Fold 1 | PIDs: ['9XOO', 'CDQ6', 'LIUY', 'MEM8', 'P3LG', 'QJ18', 'QZJ2', 'RQFN', 'TF0Y', 'UQSU', 'ZF6S'] | BA: 0.88421
  Fold 2 | PIDs: ['10ZQ', '70N8', '7FRZ', '7KPX', 'CQ2G', 'DT5C', 'NQRB', 'OL6N', 'SNG7', 'TPQI', 'VBMP', 'Y21H'] | BA: 0.94058
  Fold 3 | PIDs: ['2Q0J', '3C4K', '3H2A', '4UC1', 'BFIE', 'HDS9', 'IKYW', 'N6RZ', 'TGQ4', 'WZDL', 'ZL3U'] | BA: 0.87593
  Fold 4 | PIDs: ['01Z2', '2XO3', '9HJO', 'D1XP', 'DU2K', 'EQZH', 'EUS2', 'F1ZM', 'IYWF', 'P4DZ', 'UWUT', 'VN8D'] | BA: 0.94391
  Fold 5 | PIDs: ['13P2', '222X', '43JW', '4N6K', '7PF3', '94BI', 'C8Q6', 'J2JZ', 'MADD', 'PYXQ', 'QEYR', 'SE4Q'] | BA: 0.87150

ExtraTrees OOF BA (raw, no Stage1): 0.90251


In [19]:
# ── Find best 3-model blend ─────────────────────────────────────────────
# Grid: LGBM weight ∈ [0.0..0.6], ET weight ∈ [0.0..0.4], Cat = remainder
# Constraint: lgbm_w + et_w + cat_w = 1.0
print("\nSearching 3-model blend weights (LGBM + Cat + ET)...")
blend_rows = []
for w_lgbm in np.round(np.arange(0.0, 0.61, 0.05), 2):
    for w_et in np.round(np.arange(0.0, 0.41, 0.05), 2):
        w_cat = round(1.0 - w_lgbm - w_et, 2)
        if w_cat < 0 or w_cat > 1.0:
            continue
        blend = w_lgbm * oof_lgbm + w_cat * oof_cat + w_et * oof_et
        raw_ba = balanced_accuracy_score( y_train, np.argmax(blend, axis=1))
        blend_rows.append({'w_lgbm': w_lgbm, 'w_cat': w_cat, 'w_et': w_et, 'raw_oof_ba': raw_ba})

blend_df     = pd.DataFrame(blend_rows).sort_values('raw_oof_ba', ascending=False)
best_raw_row = blend_df.iloc[0]
print("\nTop 10 raw blend weights:")
print(blend_df.head(10).to_string(index=False))
print(f"\nBest raw weights: LGBM={best_raw_row['w_lgbm']:.2f} "
      f"Cat={best_raw_row['w_cat']:.2f} ET={best_raw_row['w_et']:.2f} "
      f"| raw OOF BA: {best_raw_row['raw_oof_ba']:.5f}")


Searching 3-model blend weights (LGBM + Cat + ET)...

Top 10 raw blend weights:
 w_lgbm  w_cat  w_et  raw_oof_ba
   0.25   0.35  0.40    0.911411
   0.25   0.40  0.35    0.911370
   0.30   0.35  0.35    0.911312
   0.30   0.45  0.25    0.911296
   0.20   0.40  0.40    0.911292
   0.25   0.50  0.25    0.911235
   0.25   0.45  0.30    0.911193
   0.30   0.30  0.40    0.911153
   0.30   0.40  0.30    0.911119
   0.35   0.35  0.30    0.910955

Best raw weights: LGBM=0.25 Cat=0.35 ET=0.40 | raw OOF BA: 0.91141


In [20]:
# ── Soft Probability Smoothing — identical to p5 ───────────────────────
def soft_prob_smooth(prob_matrix, pids, times, window=5):
    prob_matrix = np.asarray(prob_matrix)
    pids        = np.asarray(pids)
    times       = np.asarray(times)
    smoothed    = prob_matrix.copy()
    hw          = window // 2
    for pid in np.unique(pids):
        idx = np.where(pids == pid)[0]
        if len(idx) < window:
            continue
        ordered_idx  = idx[np.argsort(times[idx], kind='mergesort')]
        local_probs  = prob_matrix[ordered_idx]
        local_smooth = np.zeros_like(local_probs)
        for i in range(len(local_probs)):
            lo = max(0, i - hw)
            hi = min(len(local_probs), i + hw + 1)
            local_smooth[i] = local_probs[lo:hi].mean(axis=0)
        smoothed[ordered_idx] = local_smooth
    return smoothed

# ── p6: Tune blend weights + smoothing window jointly ──────────────────
# Search windows [3, 5, 7] in addition to the weight grid.
# Top-5 raw blend candidates are evaluated across all windows.
print("\nTuning smoothing window + Stage1 on top blend candidates...")
top_candidates = blend_df.head(15)  # evaluate top 15 raw blends with window tuning

smooth_rows = []
for _, row in top_candidates.iterrows():
    w_lgbm = row['w_lgbm']
    w_cat  = row['w_cat']
    w_et   = row['w_et']
    raw_probs_9 = w_lgbm * oof_lgbm + w_cat * oof_cat + w_et * oof_et
    score_raw   = balanced_accuracy_score(y_train, np.argmax(raw_probs_9, axis=1))

    best_window_score = -1
    best_window       = 5
    best_mode_local   = 'raw'

    for window in [3,5]:
        smooth_probs = soft_prob_smooth(
            raw_probs_9,
            groups,
            train_times,
            window=window
        )

        score_smooth = balanced_accuracy_score(
            y_train,
            np.argmax(smooth_probs, axis=1)
        )

        if score_smooth > best_window_score:
            best_window_score = score_smooth
            best_mode_local = f"smooth_w{window}"

    best_overall = max(score_raw, best_window_score)
    final_mode   = 'raw' if score_raw >= best_window_score else best_mode_local

    smooth_rows.append({
        'w_lgbm': w_lgbm, 'w_cat': w_cat, 'w_et': w_et,
        'raw_oof_ba': score_raw,
        'best_oof_ba': best_overall,
        'best_mode': final_mode,
    })

smooth_df = pd.DataFrame(smooth_rows).sort_values('best_oof_ba', ascending=False)
best      = smooth_df.iloc[0]
best_w_lgbm = float(best['w_lgbm'])
best_w_cat  = float(best['w_cat'])
best_w_et   = float(best['w_et'])
best_mode   = best['best_mode']

print("\nTop blend+window candidates:")
print(smooth_df.head(10).to_string(index=False))
print(f"\nSelected: LGBM={best_w_lgbm:.2f} Cat={best_w_cat:.2f} ET={best_w_et:.2f} "
      f"| mode: {best_mode}")



Tuning smoothing window + Stage1 on top blend candidates...

Top blend+window candidates:
 w_lgbm  w_cat  w_et  raw_oof_ba  best_oof_ba best_mode
   0.20   0.40  0.40    0.911292     0.913246 smooth_w3
   0.25   0.45  0.30    0.911193     0.912984 smooth_w3
   0.20   0.45  0.35    0.910907     0.912942 smooth_w3
   0.25   0.40  0.35    0.911370     0.912942 smooth_w3
   0.20   0.50  0.30    0.910818     0.912923 smooth_w3
   0.15   0.45  0.40    0.910894     0.912892 smooth_w3
   0.30   0.40  0.30    0.911119     0.912846 smooth_w3
   0.30   0.45  0.25    0.911296     0.912806 smooth_w3
   0.25   0.50  0.25    0.911235     0.912784 smooth_w3
   0.30   0.30  0.40    0.911153     0.912676 smooth_w3

Selected: LGBM=0.20 Cat=0.40 ET=0.40 | mode: smooth_w3


In [8]:
# ── Build final OOF and test predictions ───────────────────────────────
oof_blend_probs  = best_w_lgbm * oof_lgbm  + best_w_cat * oof_cat  + best_w_et * oof_et
test_blend_probs = best_w_lgbm * pred_lgbm + best_w_cat * pred_cat + best_w_et * pred_et

# Parse mode string to get window
def parse_mode(mode_str):

    if mode_str == 'raw':
        return 5

    if mode_str.startswith('smooth'):
        return int(mode_str.split('_w')[-1])

    return 5
best_window = parse_mode(best_mode)

if best_mode == "raw":

    final_oof_preds = np.argmax(
        oof_blend_probs,
        axis=1
    )

    test_preds_final = np.argmax(
        test_blend_probs,
        axis=1
    )

else:

    final_oof_probs = soft_prob_smooth(
        oof_blend_probs,
        groups,
        train_times,
        window=best_window
    )

    final_test_probs = soft_prob_smooth(
        test_blend_probs,
        test_label['pid'].values,
        test_times,
        window=best_window
    )

    final_oof_preds = np.argmax(
        final_oof_probs,
        axis=1
    )

    test_preds_final = np.argmax(
        final_test_probs,
        axis=1
    )
    
final_ba = balanced_accuracy_score(y_train, final_oof_preds)
print(f"\nFinal selected OOF BA: {final_ba:.5f}")
print(f"(p5 was 0.91879, p4 was 0.91132)")

NameError: name 'best_w_lgbm' is not defined

In [22]:
# ── Per-class recall vs p5 ─────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

class_names = [
    '0:JumpJack','1:Jog','2:Squat','3:MtnClimb',
    '4:PushUp','5:Burpee','6:Lunge','7:JumpSquat','8:Rest'
]
p5_recalls = [0.967, 0.943, 0.882, 0.891, 0.940, 0.942, 0.886, 0.902, 0.917]

cm = confusion_matrix(y_train, final_oof_preds)
print("\nPer-class recall (p6 vs p5):")
for i, name in enumerate(class_names):
    r     = cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0
    delta = r - p5_recalls[i]
    sign  = '+' if delta >= 0 else '-'
    print(f"  {name:20s}: {r:.3f}  {sign}{abs(delta):.3f} vs p5")



Per-class recall (p6 vs p5):
  0:JumpJack          : 0.967  -0.000 vs p5
  1:Jog               : 0.932  -0.011 vs p5
  2:Squat             : 0.863  -0.019 vs p5
  3:MtnClimb          : 0.887  -0.004 vs p5
  4:PushUp            : 0.924  -0.016 vs p5
  5:Burpee            : 0.929  -0.013 vs p5
  6:Lunge             : 0.854  -0.032 vs p5
  7:JumpSquat         : 0.897  -0.005 vs p5
  8:Rest              : 0.966  +0.049 vs p5


In [23]:
# ── Build Submission ───────────────────────────────────────────────────
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
submission.loc[has_sensor_mask, 'workout'] = test_preds_final[has_sensor_mask]
submission.loc[pid71_mask,      'workout'] = 8  # PID 71: no sensor data
submission['workout'] = submission['workout'].astype(int)

print("\nSubmission preview:")
print(submission.head(10))
print("\nPrediction distribution:")
print(submission['workout'].value_counts().sort_index())


Submission preview:
      id  workout
0  56209        8
1  56210        8
2  56211        7
3  56212        7
4  56213        7
5  56214        7
6  56215        7
7  56216        7
8  56217        7
9  56218        7

Prediction distribution:
workout
0    3840
1    3366
2    3708
3    3451
4    3916
5    3685
6    4021
7    3670
8    9816
Name: count, dtype: int64


In [24]:
# ── Save Submission ────────────────────────────────────────────────────
out_path = os.path.join(".", 'submission_p11.csv')
submission.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")
print("Done.")


Saved → ./submission_p11.csv
Done.


In [25]:
# ── Feature Importance ─────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fi_lgbm = pd.DataFrame({
    'feature':    feat_cols,
    'importance': lgbm_models[-1].feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)

fi_cat = pd.DataFrame({
    'feature':    feat_cols,
    'importance': cat_models[-1].get_feature_importance(),
}).sort_values('importance', ascending=False)

fi_et = pd.DataFrame({
    'feature':    feat_cols,
    'importance': et_models[-1].feature_importances_,
}).sort_values('importance', ascending=False)

p6_new_tags = ['acorr', 'xfft', 'inter_']
print("\nTop 30 LGBM features:")
print(fi_lgbm.head(30).to_string(index=False))

print("\nTop 30 CatBoost features:")
print(fi_cat.head(30).to_string(index=False))

print("\nTop 20 ExtraTrees features:")
print(fi_et.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 12))
plt.barh(fi_cat['feature'].head(30)[::-1], fi_cat['importance'].head(30)[::-1])
plt.xlabel('Feature importance')
plt.title(f'Top 30 Feature Importances - p6 CatBoost (OOF BA={final_ba:.5f})')
plt.tight_layout()
plt.savefig('feature_importance_p6.png', dpi=100)
plt.show()
print("Feature importance chart saved → feature_importance_p6.png")



Top 30 LGBM features:
                            feature    importance
                             gy_iqr 107763.361978
                   ayb_fft_band_low  68437.425710
                         ax_max_abs  52662.866874
                             gy_rms  42001.487184
                   gxb_fft_band_mid  39750.894658
inter_gyb_entropy_over_amag_entropy  31106.229127
                             ax_std  29849.190687
             ayb_pk_std_peak_height  29101.564643
                    gyb_fft_dom_amp  29057.959782
                   gxb_fft_dom_freq  27411.174512
                 posture_tilt_angle  24088.416403
                accel_gyro_mag_corr  22135.883839
                   gyb_xfft_dom_amp  21668.703095
                             ax_rms  18819.377374
                    ayb_fft_dom_amp  16797.349211
      inter_gyb_dom_amp_over_gy_iqr  14684.406300
                             ax_iqr  11654.084871
                   gxb_fft_band_low  11320.647372
                           